In [1]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GITHUB_PAT")
!rm -rf /kaggle/working/*
!git clone -q https://{tok}@github.com/vhrabar/SolarFilamentSegmentationChallenge.git

In [2]:
!cd SolarFilamentSegmentationChallenge && git checkout dev && git pull

Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
Already up to date.


## Train on native resolution crops

The observations are annotated at 2048, where a filament is about eleven pixels thick. Resizing one
to 512 leaves a perfect prediction with a mean IoU of 0.76 against its own annotation, and a fifth of
the filaments below the 0.5 the metric matches at. Crops keep the native scale; validation runs on
whole observations with a sliding window, so the reported PQ is the one the submission is judged on.

In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --seed 1 --device cuda:0 train \
    --arch transunet --crop 512 --crops-per-image 4 --foreground-prob 0.7 \
    --batch-size 8 --epochs 40 --lr 2e-4 --dropout 0.1 --drop-path 0.1 --warmup-epochs 3 \
    --amp --workers 4 --patience 12 \
    --val-limit 48 --val-tile 512 --val-overlap 0.0 --tile-batch 8 \
    --output-dir /kaggle/working/runs/transunet_crop512

## Calibrate the post-processing

The threshold that training picks is picked on whatever validation yields; the submission is scored on
masks at 2048. One pixel of systematic bias there costs about a quarter of the overlap with the
annotation, so the threshold, the morphological offset and the minimum area are chosen on the
2048 output itself. The winning setting is written back into the checkpoint.

In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 calibrate \
    --checkpoint /kaggle/working/runs/transunet_crop512/best.pt --tile 512 --overlap 0.25 --tile-batch 8 --workers 2 --limit 96

## Predict

The calibrated threshold, offset and minimum area come out of the checkpoint.

In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 predict \
    --checkpoint /kaggle/working/runs/transunet_crop512/best.pt --output-dir /kaggle/working/runs/transunet_crop512/predictions \
    --batch-size 1 --workers 4 --tile 512 --overlap 0.25 --tile-batch 8